In [12]:
import torch
import math
import time

In [13]:
DEVICE = "mps" if torch.mps.is_available() else "cpu"

DTYPE = torch.float32

HEAD_DIM = 64
VALUE_DIM = 64

SEQUENCE_LENGTHS = [16384,32768,65536,131072]

BLOCK_SIZES = [2048,4096,8192,16384]

WARMUP = 5
ITERATIONS = 20

In [14]:
torch.manual_seed(42)
print("="*20)
print("FLASH ATTENTION")
print("="*20)

print(f"Device: {DEVICE}")
print(f"Data Type: {DTYPE}")
print(f"Head Dimension: {HEAD_DIM}")
print(f"Value Dimension: {VALUE_DIM}")

FLASH ATTENTION
Device: mps
Data Type: torch.float32
Head Dimension: 64
Value Dimension: 64


In [15]:
def standard_attention(q,k,v):

    d = q.shape[-1]

    scores = (q @ k.T) / math.sqrt(d)

    weights = torch.softmax(scores,dim=-1)

    output = weights @ v

    return output

In [16]:
def flash_attention(q,k,v,kv_block_size=64):

    n_q, d = q.shape
    n_k = k.shape[0]

    scale = 1.0 / math.sqrt(d)

    running_max = q.new_full((n_q,1),-torch.inf)

    running_sum = q.new_zeros(n_q,1)

    accumulator = q.new_zeros(n_q,v.shape[1])

    for start in range(0,n_k,kv_block_size):

        k_tile = k[start:start+kv_block_size]
        v_tile = v[start:start+kv_block_size]

        scores = (q @ k_tile.T) * scale

        new_max = torch.maximum(
            running_max,
            scores.max(dim=-1,keepdim=True).values
        )

        old_scale = torch.exp(running_max - new_max)

        tile_weights = torch.exp(scores-new_max)

        running_sum = (
            old_scale * running_sum
            + tile_weights.sum(dim=-1,keepdim=True)
        )

        accumulator = (
            old_scale * accumulator 
            + tile_weights @ v_tile
        )

        running_max = new_max

    return accumulator / running_sum

In [17]:
def generate_data(seq_len,head_dim,value_dim,device):

    q = torch.randn(
        seq_len,
        head_dim,
        device=device,
        dtype=DTYPE
    )

    k = torch.randn(
        seq_len,
        head_dim,
        device=device,
        dtype=DTYPE
    )

    v = torch.randn(
        seq_len,
        value_dim,
        device=device,
        dtype=DTYPE
    )

    return q,k,v

In [18]:
def test_correctness(q,k,v,block_size):

    with torch.inference_mode():

        output_standard = standard_attention(q,k,v)

        output_flash = flash_attention(q,k,v,kv_block_size=block_size)

    max_error = (
        output_standard - output_flash
    ).abs().max().item()

    mean_error = (
        output_standard - output_flash
    ).abs().mean().item()

    is_correct = torch.allclose(
        output_standard,
        output_flash,
        atol=1e-5,
        rtol=1e-5
    )

    return max_error,mean_error,is_correct

In [19]:
def benchmark(fn,warmup=WARMUP,iterations=ITERATIONS):

    with torch.inference_mode():

        for _ in range(warmup):
            fn()

        if DEVICE == "cuda":
            torch.cuda.synchronize()

        start = time.perf_counter()

        for _ in range(ITERATIONS):
            fn()

        if DEVICE=="cuda":
            torch.cuda.synchronize()

        end = time.perf_counter()

    avg_time = (end-start)/iterations

    return avg_time * 1000

In [20]:
def measure_peak_memory(fn):

    if DEVICE != "cuda":
        return None

    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

    with torch.inference_mode():
        fn()
        torch.cuda.synchronize()

    peak_memory = torch.cuda.max_memory_allocated()

    return peak_memory / (1024**2)

In [21]:
def main():

    print("\n"+"="*15)
    print("STARTING BENCHMARKS")
    print("\n"+"="*15)

    for seq_len in SEQUENCE_LENGTHS:

        print(f"\n Sequence Length: {seq_len}")
        print("-"*10)

        q,k,v = generate_data(
            seq_len,
            HEAD_DIM,
            VALUE_DIM,
            DEVICE
        )

        max_error, mean_error, is_correct = test_correctness(q,k,v,BLOCK_SIZES[0])

        print("\nCORRECTNESS")
        print(f"Maximum Error: {max_error:.8f}")
        print(f"Mean Error: {mean_error:.8f}")
        print(f"Outputs Match: {is_correct}")

        if not is_correct:
            print("WARNING: Numerical mismatch detected")

        standard_time = benchmark(lambda : standard_attention(q,k,v))

        standard_memory = measure_peak_memory(lambda: standard_attention(q,k,v))

        print("\nSTANDARD ATTENTION")
        print(f"Execution Time: {standard_time:.4f} ms")

        if standard_memory is not None:
            print(f"Peak memory: {standard_memory:.4f} MB")

        print("\nFLASH ATTENTION")

        print(
            f"{'Block Size':<15}"
            f"{'Time (ms)':<15}"
            f"{'Speedup':<15}"
            f"{'Peak Memory':<15}"
        )

        for block_size in BLOCK_SIZES:

            flash_fn = lambda: flash_attention(
                q, k, v,
                kv_block_size=block_size
            )

            flash_time = benchmark(flash_fn)

            speedup = standard_time / flash_time

            flash_memory = measure_peak_memory(flash_fn)

            memory_str = (
                f"{flash_memory:.2f} MB"
                if flash_memory is not None
                else "N/A"
            )

            print(
                f"{block_size:<15}"
                f"{flash_time:<15.4f}"
                f"{speedup:<15.2f}x"
                f"{memory_str:<15}"
            )

        print("-" * 70)

In [22]:
if __name__ == "__main__":
    main()


STARTING BENCHMARKS


 Sequence Length: 16384
----------

CORRECTNESS
Maximum Error: 0.00000048
Mean Error: 0.00000002
Outputs Match: True

STANDARD ATTENTION
Execution Time: 0.1019 ms

FLASH ATTENTION
Block Size     Time (ms)      Speedup        Peak Memory    
2048           67.6746        0.00           xN/A            


KeyboardInterrupt: 